In [ ]:
import torch
import numpy as np
from pathlib import Path
from agents.ppo.stable_baseline.config import StageConfig
from agents.ppo.implementation.env import make_air_traffic_env, make_vector_env, build_observation_layout
from agents.ppo.implementation.network import AirTrafficActorCriticNetwork

In [ ]:
stage = StageConfig(
    name="stage6_ten_planes_acceleration",
    max_planes=10,
    spawn_planes=10,
    num_envs=1,
    enable_acceleration=True,
    acceleration_scale=1.0,
    enable_wind=False,
    include_wind_in_obs=False,
    total_timesteps=6_000_000,
)

model_path = Path("experiments/ppo_implementation/stage6_ten_planes_acceleration/seed_0/checkpoints/final_model.pth")
normalizer_path = Path("experiments/ppo_implementation/stage6_ten_planes_acceleration/seed_0/checkpoints/final_vec_normalize.pkl")
video_dir = Path("experiments/ppo_implementation/eval_videos")
video_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
layout = build_observation_layout(stage)
device = "cuda" if torch.cuda.is_available() else "cpu"

model = AirTrafficActorCriticNetwork(
    self_feature_dim=layout["self_feature_dim"],
    neighbor_feature_dim=layout["neighbor_feature_dim"],
    max_neighbors=layout["max_neighbors"],
    n_actions=2, 
    action_space_type="continuous"
).to(device)

model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

vec_normalize = make_vector_env(
    stage=stage,
    seed=42,
    num_envs=1,
    normalize_path=normalizer_path,
    training=False,
)

test_env = make_air_traffic_env(stage, render_mode="rgb_array")
observations, infos = test_env.reset(seed=42)
frames = []

In [ ]:
with torch.no_grad():
    while test_env.steps < 1000:
        actions = {}
        for agent in test_env.agents:
            obs = observations[agent]
            if obs[0] == -1.0:
                actions[agent] = np.zeros(test_env.action_dim, dtype=np.float32)
                continue

            normalized_obs = vec_normalize.normalize_obs(np.expand_dims(obs, axis=0))
            obs_tensor = torch.tensor(normalized_obs, dtype=torch.float32, device=device)
            
            mean, _ = model.get_action_distribution_params(obs_tensor)
            actions[agent] = mean.squeeze(0).cpu().numpy()

        observations, rewards, terminations, truncations, infos = test_env.step(actions)
        frame = test_env.render()
        if frame is not None:
            frames.append(frame)

        if all(terminations.values()) or all(truncations.values()):
            break

test_env.save_video(
    video_dir,
    frames,
    filename="eval_seed_42.mp4",
    fps=30,
)

test_env.close()
vec_normalize.close()